# MD 体系搭建流水线（E0 → M7）

配套技能：`../SKILL.md`、`../NAVIGATION.md`
分析阶段请用 `md-analysis-agent/Analysis_notebook`。

流程：环境 → 结构检查 → pdb2gmx → 拼接配体 → 盒子/溶剂化/离子 → EM → NVT → NPT → 生产

只改 **CONFIG** 一格，其余按顺序执行即可。每个 cell 都会把 gmx 输出写进 `logs_md/`。

In [ ]:
import os, glob, subprocess, shutil, sys, time

# ======================= CONFIG =======================
VARIANT   = "R279E"                       # 变体名
BASE      = "/root/autodl-tmp/GROMACS"    # 项目根
PDB_IN    = f"{BASE}/{VARIANT}/input/{VARIANT}.pdb"
OUT       = f"{BASE}/{VARIANT}/out_biop_{VARIANT}"
FF        = "amber99sb-ildn"              # 与 WT 流程一致；2024.4 不带 amber14sb.ff
WATER     = "tip3p"
BOX_D     = 1.1                           # dodecahedron 半径，1.0 底线 / 1.2 稳妥
BOX_TYPE  = "dodecahedron"                # 立方盒会多装 ~40% 的水
ION_CONC  = 0.15
CHAIN     = "A"                           # 蛋白所在链
LIG_ITP   = "PEPB.acpype/PEPB_GMX.itp"    # 配体/共轭物 itp（相对 OUT）
LIG_GRO   = "PEPB.acpype/PEPB_GMX.gro"
LIG_NAME  = "PEPB"                        # [molecules] 里的名字
NTOMP     = 12                            # 按 cgroup 配额，不看 nproc
NTMPI     = 1
# ======================================================

GMX = None
for c in ["/root/gromacs_cuda/bin/gmx", "/root/gromacs_cuda_244/bin/gmx",
          "/root/autodl-tmp/md/bin/gmx"] + glob.glob("/root/*/gromacs_cuda/bin/gmx"):
    if os.path.exists(c):
        GMX = c
        break
assert GMX, "gmx not found; run scripts/install_gromacs.sh first"
os.makedirs(f"{OUT}/mdp", exist_ok=True)
os.makedirs(f"{OUT}/logs_md", exist_ok=True)
print("GMX    :", GMX)
print("PDB_IN :", PDB_IN, os.path.exists(PDB_IN))
print("OUT    :", OUT)

In [ ]:
def run(args, stdin_text=None, tag="run", check=True):
    cmd = [GMX] + args
    log = f"{OUT}/logs_md/{tag}.log"
    r = subprocess.run(cmd, input=stdin_text, text=True, capture_output=True)
    with open(log, "w") as f:
        f.write(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"{tag} failed\n" + (r.stdout + r.stderr)[-1500:])
    return r

def read_xvg(p):
    import numpy as np
    rows = [l.split() for l in open(p) if l.strip() and l[0] not in "#@"]
    return np.array(rows, dtype=float)

## E0 环境
要看到 `GROMACS version`、`GPU support: CUDA`（生产用）。`Illegal instruction` 说明编译时 SIMD 与本机 CPU 不匹配。

In [ ]:
print(run(["--version"], tag="E0_version").stdout[:800])
print(subprocess.run(["bash", "-lc", "cat /sys/fs/cgroup/cpu.max; nvidia-smi --query-gpu=name,memory.total --format=csv,noheader"],
                     capture_output=True, text=True).stdout)

## M1 结构检查

- 只取 `CHAIN` 指定的链（配体/肽由 itp 提供，不能再从 PDB 里带进来，否则重复计数）
- 缺原子/断点用 `scripts/scan_truncated.py`、`scripts/check_breaks.py` 预检

In [ ]:
prot = f"{OUT}/prot.pdb"
with open(PDB_IN) as fi, open(prot, "w") as fo:
    for ln in fi:
        if ln.startswith("ATOM") and ln[21] == CHAIN:
            fo.write(ln)
n_atom = sum(1 for l in open(prot) if l.startswith("ATOM"))
print(f"chain {CHAIN}: {n_atom} atoms")
print(subprocess.run(["bash", "-lc", f"grep -m3 ' 279 ' {prot}"], capture_output=True, text=True).stdout)

## M3a pdb2gmx

`Total charge` 要与预期一致（本体系 WT=+6、R279E=+4）。

In [ ]:
r = run(["pdb2gmx", "-f", "prot.pdb", "-o", "prot.gro", "-p", "protA.top",
         "-ff", FF, "-water", WATER, "-ignh"], tag="M3a_pdb2gmx")
print([l for l in r.stdout.splitlines() if "Total charge" in l or "force field" in l])
print([l for l in open(f"{OUT}/protA.top") if "molecules" in l])

## M3b 拼接配体 itp

两个坑：
1. `#include` 必须放在力场 include 之后、第一个 `[ moleculetype ]` 之前——否则 `[atomtypes]` 顺序非法；
2. `[molecules]` 是文件最后一节，直接把 `LIG_NAME 1` 追加到末尾最稳。

In [ ]:
import numpy as np
INC = f'#include "{LIG_ITP}"'
lines = open(f"{OUT}/protA.top").read().splitlines()
out, done = [], False
for ln in lines:
    out.append(ln)
    if not done and "forcefield.itp" in ln:
        out += ["", INC, ""]
        done = True
assert done, "forcefield include not found"
txt = "\n".join(out).rstrip() + f"\n{LIG_NAME}                1\n"
open(f"{OUT}/topol.top", "w").write(txt)
shutil.copy(f"{OUT}/topol.top", f"{OUT}/topol.top.clean")

def read_gro(p):
    L = open(p).read().splitlines()
    n = int(L[1])
    return [l for l in L[2:2 + n]], L[2 + n].split()

a, boxa = read_gro(f"{OUT}/prot.gro")
b, _ = read_gro(f"{OUT}/{LIG_GRO}")
merged = [("%5d" % i) + l[5:] for i, l in enumerate(a + b, 1)]
box = boxa if any(float(x) > 1.0 for x in boxa) else ["4.0", "4.0", "4.0"]
with open(f"{OUT}/complex.gro", "w") as f:
    f.write(f"{VARIANT} complex\n%5d\n" % len(merged))
    f.write("\n".join(merged) + "\n")
    f.write(" ".join("%10.5f" % float(x) for x in box) + "\n")
print("complex.gro:", len(a), "+", len(b), "=", len(merged), "atoms")
print(subprocess.run(["bash", "-lc", f"sed -n '/molecules/,$p' {OUT}/topol.top"], capture_output=True, text=True).stdout)

## M3c 盒子 + 溶剂化 + 离子

`-bt dodecahedron -d 1.1`：同最小镜像距离下比立方盒省约 30% 的水。

In [ ]:
run(["editconf", "-f", "complex.gro", "-o", "box.gro", "-bt", BOX_TYPE, "-d", str(BOX_D), "-c"], tag="M3c_editconf")
run(["solvate", "-cp", "box.gro", "-cs", "spc216.gro", "-p", "topol.top", "-o", "solv.gro"], tag="M3c_solvate")
shutil.copy(f"{OUT}/topol.top", f"{OUT}/topol.top.solv")
print([l for l in open(f"{OUT}/logs_md/M3c_solvate.log") if "solvent molecules" in l])

# ions.mdp / em.mdp
MDP_EM = ("integrator=steep\nemtol=1000\nemstep=0.01\nnsteps=50000\n"
          "cutoff-scheme=Verlet\ncoulombtype=PME\nrcoulomb=1.0\nrvdw=1.0\npbc=xyz\n")
open(f"{OUT}/mdp/em.mdp", "w").write(MDP_EM)
open(f"{OUT}/mdp/ions.mdp", "w").write(MDP_EM.replace("nsteps=50000", "nsteps=500"))

run(["grompp", "-f", "mdp/ions.mdp", "-c", "solv.gro", "-p", "topol.top", "-o", "ions.tpr", "-maxwarn", "3"], tag="M3c_grompp_ions")
run(["genion", "-s", "ions.tpr", "-o", "solv_ions.gro", "-p", "topol.top",
     "-pname", "NA", "-nname", "CL", "-neutral", "-conc", str(ION_CONC)],
    stdin_text="SOL\n", tag="M3c_genion")
shutil.copy(f"{OUT}/topol.top", f"{OUT}/topol.top.solv_ions")
print([l for l in open(f"{OUT}/logs_md/M3c_genion.log") if "will be added" in l])
print(open(f"{OUT}/solv_ions.gro").readline(), open(f"{OUT}/solv_ions.gro").readlines()[1])

## M4 能量最小化
判据：`Steepest Descents converged to Fmax < 1000`。不收敛先查拓扑/接合处，不要硬跑。

In [ ]:
run(["grompp", "-f", "mdp/em.mdp", "-c", "solv_ions.gro", "-p", "topol.top", "-o", "em.tpr", "-maxwarn", "5"], tag="M4_grompp")
r = run(["mdrun", "-deffnm", "em", "-ntmpi", str(NTMPI), "-ntomp", str(NTOMP), "-nb", "gpu", "-pin", "off"], tag="M4_mdrun")
print([l for l in r.stdout.splitlines() if "converged" in l or "Maximum force" in l or "Potential Energy" in l])

## M5 NVT（100 ps，300 K）

`tc-grps = Protein Non-Protein` 与 WT 流程一致；温度应在 300 ± 5 K。

In [ ]:
MDP_NVT = """integrator=md
dt=0.002
nsteps=50000
constraint_algorithm=lincs
constraints=h-bonds
lincs_iter=1
lincs_order=4
cutoff-scheme=Verlet
coulombtype=PME
rcoulomb=1.0
rvdw=1.0
DispCorr=EnerPres
tcoupl=V-rescale
tc-grps=Protein Non-Protein
ref_t=300 300
tau_t=0.1 0.1
pcoupl=no
pbc=xyz
gen_vel=yes
gen_temp=300
gen_seed=-1
"""
open(f"{OUT}/mdp/nvt.mdp", "w").write(MDP_NVT.replace("\\n", "\n"))

run(["grompp", "-f", "mdp/nvt.mdp", "-c", "em.gro", "-p", "topol.top", "-o", "nvt.tpr", "-maxwarn", "5"], tag="M5_grompp")
r = run(["mdrun", "-deffnm", "nvt", "-ntmpi", str(NTMPI), "-ntomp", str(NTOMP),
         "-nb", "gpu", "-pme", "gpu", "-pin", "off"], tag="M5_mdrun")
print([l for l in r.stdout.splitlines() if "Performance" in l])

## M6 NPT（100 ps，C-rescale）
密度应落在约 1000 kg/m³（tip3p）。压强大幅波动是正常的。

In [ ]:
MDP_NPT = MDP_NVT.replace("nsteps=50000", "nsteps=50000").replace(
    "pcoupl=no", "pcoupl=C-rescale\npcoupltype=isotropic\ntau_p=2.0\nref_p=1.0\ncompressibility=4.5e-5")
MDP_NPT = MDP_NPT.replace("gen_vel=yes", "gen_vel=no")
open(f"{OUT}/mdp/npt.mdp", "w").write(MDP_NPT)

run(["grompp", "-f", "mdp/npt.mdp", "-c", "nvt.gro", "-t", "nvt.cpt", "-p", "topol.top", "-o", "npt.tpr", "-maxwarn", "5"], tag="M6_grompp")
r = run(["mdrun", "-deffnm", "npt", "-ntmpi", str(NTMPI), "-ntomp", str(NTOMP),
         "-nb", "gpu", "-pme", "gpu", "-pin", "off"], tag="M6_mdrun")
print([l for l in r.stdout.splitlines() if "Performance" in l])
print(subprocess.run(["bash", "-lc",
      f"printf 'Temperature\\nDensity\\n\\n' | {GMX} energy -f {OUT}/npt.edr -o /dev/null | tail -n 3"],
      capture_output=True, text=True).stdout)

## M7 生产

`-deffnm md_<V>` 只改输出名，**tpr 永远是 `md.tpr`**。
续跑：`-cpi md_<V>.cpt -nsteps <绝对总步数>`；停止用 `pkill -TERM -f "gmx mdrun"`（绝不用 -9）。

In [ ]:
MDP_MD = MDP_NPT.replace("nsteps=50000", "nsteps=50000000")
MDP_MD += "nstxout-compressed=5000\ncompressed-x-grps=System\nnstenergy=5000\nnstlog=5000\n"
open(f"{OUT}/mdp/md.mdp", "w").write(MDP_MD)
run(["grompp", "-f", "mdp/md.mdp", "-c", "npt.gro", "-t", "npt.cpt", "-p", "topol.top", "-o", "md.tpr", "-maxwarn", "5"], tag="M7_grompp")

LAUNCH = True     # 确认无误后改成 True
if LAUNCH:
    cmd = (f"cd {OUT} && nohup {GMX} mdrun -deffnm md_{VARIANT} -s md.tpr "
           f"-ntmpi {NTMPI} -ntomp {NTOMP} -nb gpu -pme gpu -pin off "
           f"> logs_md/M7_mdrun.log 2>&1 & echo pid=$!")
    print(subprocess.run(["bash", "-lc", cmd], capture_output=True, text=True).stdout)
    time.sleep(8)
    print(open(f"{OUT}/logs_md/M7_mdrun.log").read()[-600:])
else:
    print("md.tpr ready; set LAUNCH=True to start production")

## 完成后

分析用 `md-analysis-agent/analysis_agent.py`（或 `Analysis_notebook/`）：

```bash
python analysis_agent.py <VARIANT> --ref WT
```

**跨变体比较前必须确认三件事：盒子、力场、分析窗口一致。**